# 03 · At-Placement Prediction

Shows the placement information boundary and a reproducible Logistic Regression baseline, while separating that baseline from the verified saved LightGBM result.

In [ ]:
import pandas as pd
df=pd.read_csv('../data/master_orders_clean_v2.csv',low_memory=False)
train=df[df.split=='train'].copy(); test=df[df.split=='test'].copy()

## Feature boundary

Only information available when the order is created is eligible. Historical seller/product/category reputation features in the original development track were constructed chronologically so an order could not contribute its own outcome to its history.

In [ ]:
BASE_NUMERIC=['n_items','n_distinct_sellers','n_distinct_products','n_product_categories','total_price','total_freight','avg_item_price','avg_product_weight_g','max_product_weight_g','avg_product_volume_cm3','max_product_volume_cm3','total_payment_value','n_payment_installments','n_payment_methods']
PLACEMENT_FE=['purchase_hour','purchase_day_of_week','purchase_month','is_weekend','promised_delivery_days','freight_to_price_ratio','avg_freight_per_item','same_customer_seller_state']
LOW_CARD_CAT=['customer_state','primary_seller_state','primary_payment_type','primary_product_category']
HIGH_CARD_CAT=['customer_city','customer_zip_code_prefix','primary_seller_city','primary_seller_zip_code_prefix']
AT_PLACEMENT=BASE_NUMERIC+PLACEMENT_FE+LOW_CARD_CAT+HIGH_CARD_CAT
print(len(AT_PLACEMENT),'raw placement-stage features')

In [ ]:
from sklearn.model_selection import StratifiedKFold,cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

cv=StratifiedKFold(5,shuffle=True,random_state=42)
prep=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),BASE_NUMERIC+PLACEMENT_FE),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',min_frequency=20))]),LOW_CARD_CAT+HIGH_CARD_CAT)])
baseline=Pipeline([('prep',prep),('model',LogisticRegression(max_iter=2000,solver='liblinear',class_weight='balanced'))])
scores=cross_validate(baseline,train[AT_PLACEMENT],train.review_bad,cv=cv,scoring={'precision':'precision','recall':'recall','f1':'f1','auc':'roc_auc'},n_jobs=1)
{k:round(v.mean(),3) for k,v in scores.items() if k.startswith('test_')}

## Verified final result

The portfolio retains the saved held-out predictions from the final LightGBM artifact rather than silently substituting a different rerun: **precision 0.231, recall 0.511, F1 0.318, ROC-AUC 0.677**. The original development track also contains feature ablation, class-weight sensitivity and randomized hyperparameter search.